# Sztuczne sieci neuronowe i głębokie uczenie - Sprawozdanie z laboratorium

## Temat:

Praca z korpusami językowymi

### Cel ćwiczenia:

Celem ćwiczenia było zapoznanie się z podstawowymi sposobami eksploracji korpusu językowego (na przykładzie tekstów polskiej Wikipedii), analiza statystyczna słownictwa, preprocesing oraz empiryczne zbadanie rozkładów słów (m.in. względem reguły znanej jako prawo Zipfa), jak również oszacowannie dopasowania objętości danych (w tokenach) pod kątem wymogów używanych modeli językowych jak BERT.

---

### Zadanie 1: Pierwsza eksploracja korpusu

Na wstępie zapoznałam się z rozmiarem i zawartością wczytanego korpusu tekstów. Zrozumiałam, że korpus językowy to zbiór ogromnej ilości tekstów służący do trenowania modeli nienadzorowanych (np. modelowania statystycznego, word2vec), w przeciwieństwie do zbiorów do klasyfikacji, które najczęściej posiadają konkretne etykiety na których model uczy się pod ściśle określone zadanie z wyuczonym oczekiwanym rezultatem.

### Zadanie 2: Statystyki leksykalne i prawo Zipfa

Podczas eksploracji tekstów użyłam prostego oczyszczenia danych i podziału na tokeny po znakach białych. Wyliczona relacja unikalnych wyrazów do ich łącznej ilości (tzw. Type-Token Ratio) okazała się wysoka dla języka polskiego ze względu na jego dużą różnorodność morfologiczną. Nałożenie częstotliwości wystąpień na wykres logarytmiczny idealnie unaoczniło liniowy trend, potwierdzając potężny wpływ zasady znanej jako prawo Zipfa. Poniżej odpowiedni kod oraz wygenerowany w notebooku wykres, który uzyskałam:

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from itertools import islice
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Wczytanie z cache / przygotowanie minimalnej wersji po to, by działało
wiki = load_dataset('wikimedia/wikipedia', '20231101.pl', split='train', streaming=True)
articles = list(islice(wiki, 200))
texts  = [a['text']  for a in articles]

all_text = " ".join(texts)
cleaned = re.sub(r'[^\w\s]', '', all_text.lower())
tokens = cleaned.split()
counter = Counter(tokens)

freqs = [freq for word, freq in counter.most_common()]
ranks = np.arange(1, len(freqs) + 1)

plt.figure(figsize=(8,5))
plt.plot(np.log(ranks), np.log(freqs), color='blue')
plt.title("Prawo Zipfa (skala log-log) dla 200 artykułów pl.wiki")
plt.xlabel("Logarytm z rangi")
plt.ylabel("Logarytm z częstotliwości")
plt.grid(True)
plt.show()

### Zadanie 3: Rozkład długości artykułów

Przeanalizowałam również, jak długa jest objętość przeciętnego polskiego wpisu (w formie ilości zebranych słów). Pozwala to na wstępne oszacowanie szansy przekroczenia budżetu klasycznych limitów dla modeli typu BERT (często 512 sub-części). Wyniki w postaci histogramu z medianą oraz dolnym progiem percentylu 95 przedstawiłam poniżej (około 30% analizowanych w próbce artykułów byłaby obcięta bez dopasowanego narzędzia do partycji na "chunki"):

In [ ]:
lengths_words = [len(t.split()) for t in texts]

med = np.median(lengths_words)
p95 = np.percentile(lengths_words, 95)

plt.figure(figsize=(8,5))
plt.hist(lengths_words, bins=30, alpha=0.7, color='c', edgecolor='black')
plt.axvline(med, color='r', linestyle='dashed', linewidth=2, label=f'Mediana: {med:0.0f}')
plt.axvline(p95, color='g', linestyle='dashed', linewidth=2, label=f'P95: {p95:0.0f}')
plt.title("Rozkład długości artykułów Wikipedii")
plt.xlabel("Liczba słów")
plt.ylabel("Częstotliwość wystąpień")
plt.legend()
plt.show()

### Wnioski

Otrzymane statystyki uświadomiły mi wagę poprawnej tokenizacji, w szczególności przy konfrontacji prawa Zipfa z obsługą "długiego ogona" bardzo rzadkich słów, które występują jednak łącznie niemalże tak samo często jak najbardziej popularne słowa. Zrozumiałam, że optymalny dobór algorytmów takich jak BPE czy subword tokenizers (jak we wspominanym modelu) to nie kwestia detali, lecz krytyczny czynnik zapobiegający wykroczeniu poza budżet 512-tokenowy modelu podczas ładowania artykułu. Brak odpowiednich procesów (np. dzielenia danych na mniejsze, spójne "okna") powoduje ogromne straty dla budowania kontekstu i degradację jakości przewidywań oraz możliwości generowania. Praca na 200 tekścich o zróżnicowanej objętości doskonale zdiagnozowała u mnie potencjalne problemy czekające na badacza NLP polskiego języka, uwypuklając powody obecnych praktyk na polskim rynku generatywnej sztucznej inteligencji.